# The designated outcome: its account and its contrasts

Designating a target tells the profiler which column the
table is about. Two passes then state the outcome itself
before anything about its drivers. The account
(`outcome_account`) states the outcome's own distribution:
for a numeric outcome its median, mean, middle half, and
range, in the outcome's declared unit when one is declared;
for a categorical outcome the share of each level, and the
declared event's rate. The contrasts (`outcome_contrasts`)
state how the outcome varies across the levels of each
categorical grouping column, against the pooled outcome —
the comparison the per-level contrast scan does not make,
because it compares levels on measures with no outcome
designated.

Twelve hundred bookings arrive through three channels. The
`direct` channel cancels at twice the rate of the other two;
the region the booking came from makes no difference. The
caller designates the outcome, names the event, and declares
what the data is for.

In [1]:
import polars as pl
import onceover

channels, regions, canceled = [], [], []
for channel, rate in [("direct", 0.6), ("agent", 0.3), ("partner", 0.3)]:
    for i in range(400):
        channels.append(channel)
        regions.append("north" if (len(canceled) % 2 == 0) else "south")
        canceled.append("yes" if i < rate * 400 else "no")
bookings = pl.DataFrame(
    {"channel": channels, "region": regions, "canceled": canceled}
)

report = onceover.profile(
    bookings,
    target="canceled",
    hints={"event": "yes", "purpose": "understanding cancellation"},
)
for finding in report.findings:
    if finding["type"] in ("outcome_distribution", "outcome_contrast"):
        print(finding["claim"])
print()
print(report.coverage["outcome_contrasts"]["detail"])

No difference in the 'yes' rate across 'region' was measured: 'north' 40.0% of 600, 'south' 40.0% of 600, against the pooled 40.0% (rows carrying 'yes' or 'no').
channel = 'direct': the 'yes' rate is 60.0% of 400 rows carrying 'yes' or 'no', against the pooled 40.0%.
channel = 'agent': the 'yes' rate is 30.0% of 400 rows carrying 'yes' or 'no', against the pooled 40.0%.
channel = 'partner': the 'yes' rate is 30.0% of 400 rows carrying 'yes' or 'no', against the pooled 40.0%.
The designated outcome 'canceled' takes 2 levels over 1200 rows: 'no' 60.0% (720 rows), 'yes' 40.0% (480 rows). The declared event is 'yes': 40.0% of all rows, and 40.0% of the 1200 rows carrying 'yes' or 'no'.

target 'canceled' designated (categorical, event 'yes'); basis: the 1200 rows carrying 'yes' or 'no', pooled event rate 40.0%; 2 groupings measured, 1 entered, 1 nulls stated, 0 measured without a statement; 4 statements stated


## Reading the result

Rates for a categorical outcome are computed on the event
basis: the rows carrying the event level or the reference
level — the target's modal other level, which for a binary
outcome is every row. The basis is named in every claim, and
the account also states the event's share of all rows, so a
rate is never read against the wrong denominator. A grouping
is eligible with at least two levels of 30 or more rows; each
grouping is tested by an omnibus chi-square and each level
by a two-proportion z-test against the pooled rate, the two
families corrected separately by Benjamini–Hochberg. A
grouping enters when its omnibus q-value is at most 0.05 and
at least one level differs from the pooled rate by two
percentage points or more; an entered grouping states every
eligible level, marking each as a measured difference or as
beside the pooled rate. For a numeric outcome the same shape
holds with medians: a level enters at half the pooled IQR
against the pooled median, by rank-sum test. Without a
declared event, a categorical outcome's contrasts take a
weaker per-level form — each level's share of each outcome
level against that level's pooled share.

The `region` statement is a null, stated at the same
prominence as the channel differences: no difference was
measured, and the measurement was precise enough to have
seen one. A categorical null requires the omnibus p-value at
least 0.10, every level's 95% interval for its rate to
contain the pooled rate, and every interval's half-width at
most five percentage points; a numeric null requires every
level's median within a quarter of the pooled IQR. Nulls are
stated only in the pointed mode — when the caller declared
metadata alongside the target. On an undeclared run a null
would fire on any flat grouping, including groupings whose
flatness carries no information for a reader who declared
nothing, so the same profile without the declaration states
the differences and withholds the null:

In [2]:
bare = onceover.profile(bookings, target="canceled")
for finding in bare.findings:
    if finding["type"] == "outcome_contrast":
        print(finding["claim"])
print()
print(bare.coverage["outcome_contrasts"]["detail"])

channel = 'direct': 'no' is 40.0% of 400 rows, against 60.0% of all rows.
channel = 'agent': 'yes' is 30.0% of 400 rows, against 40.0% of all rows.
channel = 'partner': 'yes' is 30.0% of 400 rows, against 40.0% of all rows.

target 'canceled' designated (categorical, no event declared); 2 groupings measured, contrasts stated per level of the outcome against its pooled share; absences are not stated: without a declared event there is no single rate whose absence is the statement; 6 statements stated


Grouping columns recorded after the outcome (the
`recorded_after` hint) never enter the scan — their levels
would restate the outcome — and the coverage entry states
the exclusions. The two passes run only on a designated
target; naming them in `detectors=[...]` without one records
them as skipped.